<a href="https://colab.research.google.com/github/ZuhaaAsif/Applied-Search-Intelligence-Google-Search-Ranking-Discoverability/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZuhaaAsif/Applied-Search-Intelligence-Google-Search-Ranking-Discoverability/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Ranking / Scoring. The core question "which pages should a reviewer check first?" maps directly to "Which ones first?". It's not classification, because there's no clean yes/no outcome yet; not clustering, because I'm not grouping pages into types, I'm ordering them by urgency.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target/proxy: CTR gap, a page's actual CTR minus its position tier's average CTR, computed only on pages with real position data (avg_position > 0; excludes the "no data" placeholder) and enough volume (impressions_90d >= 100) to avoid noise. This is a directly observed, computed quantity, not a rule someone already decided.

Two things about this target worth flagging: 27.6% of position/volume-filtered pages (6,076 of 22,006) have ctr exactly 0 despite meeting the volume threshold; a large block where the gap score can't differentiate, since they're all tied at the same minimum. Separately, mean CTR is slightly higher for page_1 (0.355) than top_3 (0.334), counter to a naive best-to-worst assumption.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

scored = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()
tier_avg_ctr = scored.groupby("position_tier")["ctr"].mean()
print(tier_avg_ctr.round(4).to_string())

position_tier
deep        0.0554
page_1      0.3548
page_3_5    0.1424
striking    0.2558
top_3       0.3341


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@K via manual top-K review: Since this is an unsupervised score with no ground-truth label yet, good is checked the way the validation guide recommends for ranked queues: read the actual top-ranked pages. Of the top 20 pages by CTR gap, good means at least ~80% genuinely show real position data, sufficient volume, and a CTR meaningfully below their tier average with no obvious artifact.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k_manual_flag(df_sorted, k, valid_mask):
    top_k = df_sorted.head(k)
    return valid_mask.loc[top_k.index].mean()

# sanity check: by construction, every row already passed the position/volume filter —
# this becomes a real audit once "valid_mask" reflects actual manual review, not just the filter
sanity_mask = (scored["avg_position"] > 0) & (scored["impressions_90d"] >= 100)
print(f"Precision@20 (sanity check only): {precision_at_k_manual_flag(scored.sort_values('ctr'), 20, sanity_mask):.3f}")

Precision@20 (sanity check only): 1.000


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page. The printed top-10 by ctr_gap looks like a ranking but reveals a structural issue: every row has ctr exactly 0.0 and an identical gap (-0.355), because 6,076 of 22,006 qualifying pages (27.6%) share that exact value, top 10 here is an arbitrary slice of a large tied block, not a genuine ranking. That's a real finding on its own but it means the gap score needs a tiebreaker before it's a working ranking.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
scored["ctr_gap"] = scored["ctr"] - scored.groupby("position_tier")["ctr"].transform("mean")
scored_sorted = scored.sort_values("ctr_gap")
scored_sorted[["content_id", "position_tier", "ctr", "avg_position", "impressions_90d", "ctr_gap"]].head(10)

,content_id,position_tier,ctr,avg_position,impressions_90d,ctr_gap
27776,content_7f0efbee72d5,page_1,0.0,6.3,137,-0.35476
14937,content_06b02c9b01a1,page_1,0.0,5.3,328,-0.35476
6648,content_10d038b1695e,page_1,0.0,9.7,599,-0.35476
20021,content_941418657a71,page_1,0.0,6.6,253,-0.35476
6717,content_942399f999b7,page_1,0.0,7.6,858,-0.35476
6726,content_6abe54161f91,page_1,0.0,9.0,579,-0.35476
28791,content_5053a59f5e7e,page_1,0.0,8.3,258,-0.35476
18696,content_05c9843a9494,page_1,0.0,4.6,492,-0.35476
5353,content_847c0e9b2eb6,page_1,0.0,6.9,328,-0.35476
2582,content_9648b7053d6f,page_1,0.0,9.3,6524,-0.35476


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A flat rule is already wrong, since expected CTR varies enormously by position. The tier-mean adjustment fixes that one variable, but it's still a two-variable rule. The real question is whether other signals shift what expected CTR should be within a tier, in ways a single hand-written adjustment can't capture. The code below checks that directly: if expected CTR varies meaningfully across content types even inside the same tier, that's a real, tangled, multi-signal pattern.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cross = scored.groupby(["position_tier", "content_type"])["ctr"].mean().unstack()
print(cross.round(4))

content_type   comparison article  feedly article  keyword article
position_tier                                                     
deep                          NaN          0.0440           0.0555
page_1                     0.1412          0.9048           0.3458
page_3_5                   0.0940          0.1656           0.1427
striking                   0.1474          0.3580           0.2559
top_3                      0.0000          2.9000           0.3104


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.